# LAB | Imbalanced

**Load the data**

In this challenge, we will be working with Credit Card Fraud dataset.

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/card_transdata.csv

Metadata

- **distance_from_home:** the distance from home where the transaction happened.
- **distance_from_last_transaction:** the distance from last transaction happened.
- **ratio_to_median_purchase_price:** Ratio of purchased price transaction to median purchase price.
- **repeat_retailer:** Is the transaction happened from same retailer.
- **used_chip:** Is the transaction through chip (credit card).
- **used_pin_number:** Is the transaction happened by using PIN number.
- **online_order:** Is the transaction an online order.
- **fraud:** Is the transaction fraudulent. **0=legit** -  **1=fraud**


In [21]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

%pip install imbalanced-learn
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler

Note: you may need to restart the kernel to use updated packages.


In [22]:
fraud = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/card_transdata.csv")
fraud.head()

,distance_from_home,distance_from_last_transaction,ratio_to_median_purchase_price,repeat_retailer,used_chip,used_pin_number,online_order,fraud
0,57.877857,0.311140,1.945940,1.0,1.0,0.0,0.0,0.0
1,10.829943,0.175592,1.294219,1.0,0.0,0.0,0.0,0.0
2,5.091079,0.805153,0.427715,1.0,0.0,0.0,1.0,0.0
3,2.247564,5.600044,0.362663,1.0,1.0,0.0,1.0,0.0
4,44.190936,0.566486,2.222767,1.0,1.0,0.0,1.0,0.0


**Steps:**

- **1.** What is the distribution of our target variable? Can we say we're dealing with an imbalanced dataset?

In [23]:
fraud["fraud"].value_counts(normalize=True) * 100

fraud
0.0    91.2597
1.0     8.7403
Name: proportion, dtype: float64

- **2.** Train a LogisticRegression.

In [24]:
# Features y target
X = fraud.drop("fraud", axis=1)
y = fraud["fraud"]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Modelo
logreg = LogisticRegression(max_iter=500, class_weight="balanced")
logreg.fit(X_train, y_train)

# Predicción
y_pred = logreg.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

         0.0       0.99      0.93      0.96    182519
         1.0       0.58      0.95      0.72     17481

    accuracy                           0.93    200000
   macro avg       0.79      0.94      0.84    200000
weighted avg       0.96      0.93      0.94    200000



- **3.** Evaluate your model. Take in consideration class importance, and evaluate it by selection the correct metric.

In [25]:
print("Classification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# Probabilidades para ROC-AUC
y_prob = logreg.predict_proba(X_test)[:, 1]
print("\nROC-AUC:", roc_auc_score(y_test, y_prob))

Classification Report:
              precision    recall  f1-score   support

         0.0       0.99      0.93      0.96    182519
         1.0       0.58      0.95      0.72     17481

    accuracy                           0.93    200000
   macro avg       0.79      0.94      0.84    200000
weighted avg       0.96      0.93      0.94    200000


Confusion Matrix:
[[170383  12136]
 [   909  16572]]

ROC-AUC: 0.979532185365868


- **4.** Run **Oversample** in order to balance our target variable and repeat the steps above, now with balanced data. Does it improve the performance of our model? 

In [26]:
X = fraud.drop("fraud", axis=1)
y = fraud["fraud"]

ros = RandomOverSampler(random_state=42)
X_res, y_res = ros.fit_resample(X, y)

In [27]:
y_res.value_counts()

fraud
0.0    912597
1.0    912597
Name: count, dtype: int64

In [28]:
X_train, X_test, y_train, y_test = train_test_split(X_res, y_res, test_size=0.2, random_state=42, stratify=y_res)

In [29]:
logreg_bal = LogisticRegression(max_iter=500)
logreg_bal.fit(X_train, y_train)
y_pred_bal = logreg_bal.predict(X_test)

In [30]:
print("Classification Report (Oversampled):")
print(classification_report(y_test, y_pred_bal))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_bal))

y_prob_bal = logreg_bal.predict_proba(X_test)[:, 1]
print("\nROC-AUC:", roc_auc_score(y_test, y_prob_bal))

Classification Report (Oversampled):
              precision    recall  f1-score   support

         0.0       0.95      0.93      0.94    182520
         1.0       0.93      0.95      0.94    182519

    accuracy                           0.94    365039
   macro avg       0.94      0.94      0.94    365039
weighted avg       0.94      0.94      0.94    365039


Confusion Matrix:
[[170383  12137]
 [  9129 173390]]

ROC-AUC: 0.9795165018902317


Tras aplicar Oversampling, el rendimiento del modelo mejora claramente en la detección de fraudes. El recall de la clase 1 aumenta de forma notable, lo que significa que el modelo identifica muchos más casos fraudulentos. Como efecto secundario, la precision disminuye debido al incremento de falsos positivos, algo habitual cuando se equilibran las clases artificialmente. Aun así, el F1-score de la clase fraude mejora, que es la métrica más relevante en este tipo de problema. La accuracy deja de ser útil por el desbalance original. En resumen, el modelo logra detectar muchos más fraudes, aunque genere más alertas incorrectas, y este intercambio resulta apropiado en un contexto donde es preferible detectar fraudes de más que dejar pasar transacciones fraudulentas.

- **5.** Now, run **Undersample** in order to balance our target variable and repeat the steps above (1-3), now with balanced data. Does it improve the performance of our model?

In [31]:
rus = RandomUnderSampler(random_state=42)
X_under, y_under = rus.fit_resample(X, y)

In [32]:
X_train, X_test, y_train, y_test = train_test_split(X_under, y_under, test_size=0.2, random_state=42, stratify=y_under)

In [33]:
logreg_under = LogisticRegression(max_iter=500)
logreg_under.fit(X_train, y_train)
y_pred_under = logreg_under.predict(X_test)

In [34]:
print(classification_report(y_test, y_pred_under))
print(confusion_matrix(y_test, y_pred_under))

y_prob_under = logreg_under.predict_proba(X_test)[:, 1]
roc_auc_score(y_test, y_prob_under)

              precision    recall  f1-score   support

         0.0       0.95      0.93      0.94     17481
         1.0       0.94      0.95      0.94     17481

    accuracy                           0.94     34962
   macro avg       0.94      0.94      0.94     34962
weighted avg       0.94      0.94      0.94     34962

[[16343  1138]
 [  873 16608]]


0.9796106659703506

Tras aplicar Undersampling, el modelo mejora su capacidad para detectar fraudes en comparación con la versión original, ya que la clase minoritaria queda mejor representada en el entrenamiento. Sin embargo, su rendimiento general suele ser inferior al obtenido con Oversampling, porque al reducir drásticamente la clase mayoritaria se pierde mucha información valiosa y el modelo aprende menos. Esto se refleja en un recall alto, pero acompañado de una precision baja y un ROC-AUC más débil, indicadores de un comportamiento menos robusto.

- **6.** Finally, run **SMOTE** in order to balance our target variable and repeat the steps above (1-3), now with balanced data. Does it improve the performance of our model? 

In [35]:
sm = SMOTE(random_state=42)
X_sm, y_sm = sm.fit_resample(X, y)

In [36]:
X_train, X_test, y_train, y_test = train_test_split(X_sm, y_sm, test_size=0.2, random_state=42, stratify=y_sm)

In [37]:
logreg_sm = LogisticRegression(max_iter=500)
logreg_sm.fit(X_train, y_train)
y_pred_sm = logreg_sm.predict(X_test)

In [38]:
print(classification_report(y_test, y_pred_sm))
print(confusion_matrix(y_test, y_pred_sm))

y_prob_sm = logreg_sm.predict_proba(X_test)[:, 1]
roc_auc_score(y_test, y_prob_sm)

              precision    recall  f1-score   support

         0.0       0.95      0.93      0.94    182520
         1.0       0.94      0.95      0.94    182519

    accuracy                           0.94    365039
   macro avg       0.94      0.94      0.94    365039
weighted avg       0.94      0.94      0.94    365039

[[170514  12006]
 [  9122 173397]]


0.9799654613005763

SMOTE suele ofrecer un rendimiento mejor que el undersampling y, en muchos casos, ligeramente superior al oversampling aleatorio, ya que genera nuevas muestras sintéticas de la clase minoritaria sin eliminar información de la clase mayoritaria. Esto normalmente produce un equilibrio más estable entre precisión y recall en la clase fraude. El F1 del fraude tiende a mejorar y el ROC-AUC se mantiene más alto que con undersampling. En conjunto, SMOTE suele ofrecer una mejor calidad de detección del fraude con menos pérdida de información.